# Matter–Antimatter Behaviour of Every SME Fermion Coefficient

**MSc Research — Oyewo Temidayo Solomon, University of Ibadan**

This notebook fixes, for every coefficient family of the minimal SME fermion
sector, how the antiparticle's non-relativistic Hamiltonian relates to the
particle's. It is the verification behind the final column of the master
matching table (Chapter 4) and the antiparticle statements in Sections 4.1.1,
4.1.2 and 4.1.4.

Conventions follow `dirac_algebra.py`: Dirac representation, metric $(+,-,-,-)$,
$\hbar=c=1$. The Lagrangian and the kinetic-sector field redefinition
$A = 1-\tfrac12\gamma^0(\Gamma_0-\gamma_0)$ follow Kostelecký & Lane,
*J. Math. Phys.* **40**, 6245 (1999), Eqs. (1)–(3).

## 1. Three possible comparisons

After the Foldy–Wouthuysen (FW) reduction, the even Hamiltonian splits into an upper block $h_{\rm up}(\mathbf p,\boldsymbol\sigma)$ (particle) and a lower block $h_{\rm low}(\mathbf p,\boldsymbol\sigma)$ (negative-energy solutions). In hole theory, an antiparticle of momentum $\mathbf p$ and spin $\mathbf s$ is the absence of a negative-energy solution of momentum $-\mathbf p$ and spin $-\mathbf s$, with the opposite energy. Writing $h_{\rm low}=a_0+\mathbf a\cdot\boldsymbol\sigma$, this is the charge-conjugate ($\chi\to i\sigma_2\chi^*$) statement

$$h^{\bar f}_{\text{same spin}}(\mathbf p,\boldsymbol\sigma) = -a_0(-\mathbf p) + \mathbf a(-\mathbf p)\cdot\boldsymbol\sigma .$$

CPT takes a particle $(\mathbf p,\mathbf s)$ to an antiparticle $(\mathbf p,-\mathbf s)$. Comparing CPT-conjugate states therefore uses

$$h^{\bar f}_{\text{CPT-conj}}(\mathbf p,\boldsymbol\sigma) = -h_{\rm low}(-\mathbf p,\boldsymbol\sigma).$$

This is the comparison under which exact CPT symmetry gives $g^{\bar f}_\alpha=g^f_\alpha$, and hence $A_\alpha=0$.

A third construction, $-h_{\rm low}(\mathbf p,\boldsymbol\sigma)$, flips the energy but reverses neither momentum nor spin. It is not a physical comparison. It is shown only because an earlier version of the thesis used it. It agrees with the CPT-conjugate comparison only for terms even in $\mathbf p$.

Expected results:
* **CPT-conjugate comparison**: the coefficient's CPT parity (odd $\Rightarrow$ flip, even $\Rightarrow$ same).
* **Same-spin comparison**: Kostelecký & Lane's antiparticle rule (J. Math. Phys. 40, 6245, final paragraph): $a\to-a,\ b\to+b,\ c\to+c,\ d\to-d,\ e\to-e,\ f\to-f,\ g\to+g,\ H\to-H$.

## 2. FW reduction for an arbitrary coefficient family

In [1]:
# Full check: FW reduction (K&L field redefinition) for every SME coefficient family,
# then compare particle vs antiparticle by (i) the thesis's "-lower block at same p,sigma"
# and (ii) proper charge conjugation: h_anti(p) = -[a0(-p) - a(-p).sigma] for h_lower = a0 + a.sigma.
import sys, os
sys.path.insert(0, os.getcwd())
import sympy as sp
from sympy import I, Rational, eye, zeros, Matrix
from dirac_algebra import beta, gamma, gamma5, sigma_munu, even_part, odd_part, upper, lower, bcf_expansion
I4=eye(4); p=sp.symbols('p1:4', real=True); m=sp.symbols('m', positive=True)
eps,lam=sp.symbols('eps lam', positive=True)
gl=[gamma[0],-gamma[1],-gamma[2],-gamma[3]]           # gamma_mu (lower)
def tr(M): return Matrix(M).applyfunc(lambda x: sp.expand(sp.series(sp.expand(x),eps,0,2).removeO()))
def lead(M):
    M=Matrix(M).subs({pi:lam*pi for pi in p},simultaneous=True)
    return M.applyfunc(lambda x: sp.expand(sp.series(sp.expand(x),lam,0,2).removeO()).subs(lam,1))
S=lambda n: sp.Symbol(n, real=True)
def build(fam):
    M=m*I4; G=[gl[n] for n in range(4)]
    k,idx=fam
    if k=='a':  M+= eps*sum((S(f'a{mu}')*gl[mu] for mu in idx), zeros(4,4))                   # a_mu gamma^mu = a^mu gamma_mu
    if k=='b':  M+= eps*sum((S(f'b{mu}')*gamma5*gl[mu] for mu in idx), zeros(4,4))
    if k=='H':  M+= eps*sum((S(f'H{a}{b}')*sigma_munu(a,b) for a,b in idx), zeros(4,4))        # (1/2)H sigma, antisym -> sum a<b
    for (mu,nu) in (idx if k in 'cd' else []):
        X=gamma[mu] if k=='c' else gamma5*gamma[mu]
        G[nu]=G[nu]+eps*S(f'{k}{mu}{nu}')*X
    for nu in (idx if k in 'ef' else []):
        G[nu]=G[nu]+eps*S(f'{k}{nu}')*(I4 if k=='e' else I*gamma5)
    for (l,mu,nu) in (idx if k=='g' else []):
        G[nu]=G[nu]+eps*S(f'g{l}{mu}{nu}')*sigma_munu(l,mu)
    dG0=G[0]-gl[0]; A=tr(I4-Rational(1,2)*gamma[0]*dG0); Ab=tr(gamma[0]*A.H*gamma[0])
    H=gamma[0]*tr(Ab*M*A)
    for j in (1,2,3): H=H-gamma[0]*tr(Ab*G[j]*A)*p[j-1]
    H=tr(H); herm=sp.simplify(tr(H.H-H))==zeros(4,4)
    O=odd_part(H); Sfw=-I*beta*O/(2*m); _,tot=bcf_expansion(Sfw,H,order=2)
    E=even_part(tr(tot))
    strip=lambda B: lead(sp.expand(Matrix(B))-sp.expand(Matrix(B).subs(eps,0))).subs(eps,1)
    return herm, strip(upper(E)), strip(lower(E))


## 3. Comparison for every family

In [2]:
sx,sy,sz = Matrix([[0,1],[1,0]]), Matrix([[0,-I],[I,0]]), Matrix([[1,0],[0,-1]])
def dec(h): return h.trace()/2, [(h*s).trace()/2 for s in (sx,sy,sz)]
def verdict(ha, hp):
    if sp.simplify(sp.expand(ha-hp)) == zeros(2,2): return 'SAME'
    if sp.simplify(sp.expand(ha+hp)) == zeros(2,2): return 'FLIP'
    return 'mixed'
flipP = lambda M: M.subs({pi:-pi for pi in p}, simultaneous=True)

sp3=[1,2,3]
fams = {
 'a_0':('a',[0]), 'a_j':('a',sp3), 'b_0':('b',[0]), 'b_j':('b',sp3),
 'H_ij':('H',[(1,2),(1,3),(2,3)]), 'H_0i':('H',[(0,1),(0,2),(0,3)]),
 'c_00':('c',[(0,0)]), 'c_0j,c_j0':('c',[(0,j) for j in sp3]+[(j,0) for j in sp3]),
 'd_00':('d',[(0,0)]), 'd_j0':('d',[(j,0) for j in sp3]), 'd_0j':('d',[(0,j) for j in sp3]),
 'd_jk':('d',[(j,k) for j in sp3 for k in sp3]),
 'e_0':('e',[0]), 'e_j':('e',sp3), 'f_0':('f',[0]), 'f_j':('f',sp3),
 'g_[kl]0':('g',[(1,2,0),(1,3,0),(2,3,0)]), 'g_[0k]0':('g',[(0,k,0) for k in sp3]),
 'g_[kl]j':('g',[(a,b,j) for a,b in [(1,2),(1,3),(2,3)] for j in sp3]),
}
CPT = {'a':'FLIP','b':'FLIP','c':'SAME','d':'SAME','e':'FLIP','f':'FLIP','g':'FLIP','H':'SAME'}
KL  = {'a':'FLIP','b':'SAME','c':'SAME','d':'FLIP','e':'FLIP','f':'FLIP','g':'SAME','H':'FLIP'}

rows, checks = [], []
for name, fam in fams.items():
    herm, hp, hl = build(fam)
    if hp == zeros(2,2):
        rows.append((name, herm, '-', '(no term at this order)', '-', '-')); continue
    a0, a = dec(flipP(hl))
    same_spin = -(a0*eye(2) - (a[0]*sx + a[1]*sy + a[2]*sz))
    cpt_conj  = -flipP(hl)
    spin = any(sp.simplify(x) != 0 for x in dec(hp)[1])
    v_cpt, v_same = verdict(cpt_conj, hp), verdict(same_spin, hp)
    checks.append((name, v_cpt == CPT[fam[0]], v_same == KL[fam[0]]))
    rows.append((name, herm, 'yes' if spin else 'no', verdict(-hl, hp), v_cpt, v_same))

print(f"{'family':11s}{'Hermitian':10s}{'spin':6s}{'-h_low(p,s) [old]':22s}{'CPT-conjugate':15s}same spin")
for r in rows: print(f"{r[0]:11s}{str(r[1]):10s}{r[2]:6s}{r[3]:22s}{r[4]:15s}{r[5]}")
print()
print("CPT-conjugate comparison equals CPT parity in every family:", all(c[1] for c in checks))
print("Same-spin comparison equals Kostelecky-Lane rule in every family:", all(c[2] for c in checks))

family     Hermitian spin  -h_low(p,s) [old]     CPT-conjugate  same spin
a_0        True      no    FLIP                  FLIP           FLIP
a_j        True      no    SAME                  FLIP           FLIP
b_0        True      yes   SAME                  FLIP           SAME
b_j        True      yes   FLIP                  FLIP           SAME
H_ij       True      yes   SAME                  SAME           FLIP
H_0i       True      yes   FLIP                  SAME           FLIP
c_00       True      no    SAME                  SAME           SAME
c_0j,c_j0  True      no    FLIP                  SAME           SAME
d_00       True      yes   FLIP                  SAME           FLIP
d_j0       True      yes   SAME                  SAME           FLIP
d_0j       True      -     (no term at this order)-              -
d_jk       True      yes   FLIP                  SAME           FLIP
e_0        True      no    FLIP                  FLIP           FLIP
e_j        True      no    SAME

## 4. Result

* In the **CPT-conjugate comparison**, every family follows its coefficient's CPT parity. The spin-dependent families of $b_\mu$ and $g_{\lambda\mu\nu}$ (CPT-odd) all flip, and those of $H_{\mu\nu}$ and $d_{\mu\nu}$ (CPT-even) all keep their sign. This is the final column of the master matching table.
* In the **same-spin comparison**, every family follows Kostelecký & Lane's antiparticle rule. For spin-linear terms it is the opposite of the CPT-conjugate result, because such a term changes sign when the spin is reversed.
* The **old construction** $-h_{\rm low}(\mathbf p,\boldsymbol\sigma)$ agrees with the CPT-conjugate comparison only for terms even in $\mathbf p$. It fails for $a_j$, $b_0$, $H_{0i}$, $c_{0j}$, $d_{00}$, $d_{jk}$, $e_j$, $g_{[0k]0}$ and $g_{[kl]j}$. That failure is why an earlier version of the thesis reported that only one $g_{\lambda\mu\nu}$ family flips.
* The spin-independent families ($a$, $c$, $e$) are an independent control: their two comparisons coincide, as they must, and both match the published rule.